<a href="https://colab.research.google.com/github/sagunadk07/TVISHA-Skin-analysis/blob/main/dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from PIL import Image
import timm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report


In [4]:
from google.colab import drive
drive.mount("/content/drive/")
# =========================
# CONFIGURATION
# =========================
DATA_ROOT = "/content/drive/MyDrive/Tvisha/dataset"          # CHANGE THIS
SAVE_PATH = "/content/drive/MyDrive/Tvisha/dataset/best_skin_model_8class.pth"
PLOTS_DIR = "training_plots"                         # folder to save graphs
os.makedirs(PLOTS_DIR, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 50
LR = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
DROPOUT_RATE = 0.4
PATIENCE = 8

TRAIN_RATIO = 0.7
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Automatically read class names from subfolders
CLASS_NAMES = sorted([d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))])
NUM_CLASSES = len(CLASS_NAMES)
print(f"Found classes: {CLASS_NAMES} ({NUM_CLASSES} classes)")

Mounted at /content/drive/
Found classes: ['Redness', 'dark spots', 'inflammatory acne', 'non inflammatory acne black heads', 'non inflammatory acne white heads', 'pigmentation', 'pores', 'wrinkles'] (8 classes)


In [5]:
os.makedirs("/content/drive/MyDrive/Tvisha/Model", exist_ok=True)

SAVE_PATH = "/content/drive/MyDrive/Tvisha/Model/best_skin_model_8class.pth"

In [6]:
from torchvision import datasets

dataset = datasets.ImageFolder(DATA_ROOT)

print("Classes:", dataset.classes)
print("Total Images:", len(dataset))

Classes: ['Redness', 'dark spots', 'inflammatory acne', 'non inflammatory acne black heads', 'non inflammatory acne white heads', 'pigmentation', 'pores', 'wrinkles']
Total Images: 4286


In [7]:
from collections import Counter
from torchvision import datasets

dataset = datasets.ImageFolder(DATA_ROOT)

counts = Counter(dataset.targets)

for idx, count in counts.items():
    print(f"{dataset.classes[idx]}: {count}")

Redness: 600
dark spots: 669
inflammatory acne: 606
non inflammatory acne black heads: 312
non inflammatory acne white heads: 300
pigmentation: 599
pores: 600
wrinkles: 600


In [8]:
# =========================
# DATA TRANSFORMS
# =========================
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.GaussianBlur(kernel_size=(3,3), sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [9]:
# =========================
# LOAD DATASET AND SPLIT
# =========================
DATA_ROOT= "/content/drive/MyDrive/Tvisha/dataset"
full_dataset = datasets.ImageFolder(root=DATA_ROOT, transform=None)
total_len = len(full_dataset)
train_len = int(TRAIN_RATIO * total_len)
val_len   = int(VAL_RATIO * total_len)
test_len  = total_len - train_len - val_len

train_subset, val_subset, test_subset = random_split(full_dataset, [train_len, val_len, test_len])

class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
    def __getitem__(self, index):
        img, label = self.subset[index]
        if self.transform:
            img = self.transform(img)
        return img, label
    def __len__(self):
        return len(self.subset)

train_dataset = TransformSubset(train_subset, transform=train_transforms)
val_dataset   = TransformSubset(val_subset,   transform=val_test_transforms)
test_dataset  = TransformSubset(test_subset,  transform=val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Train: 3000 | Val: 642 | Test: 644


In [10]:
# =========================
# MODEL
# =========================
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=NUM_CLASSES)
in_features = model.classifier.in_features
model.classifier = nn.Sequential(
    nn.Dropout(DROPOUT_RATE),
    nn.Linear(in_features, NUM_CLASSES)
)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

In [11]:
# =========================
# TRAINING & EVAL FUNCTIONS
# =========================
def train_one_epoch():
    model.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

@torch.no_grad()
def evaluate_accuracy(loader):
    model.eval()
    correct = 0
    total = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model(x)
        preds = torch.argmax(out, dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    return correct / total

@torch.no_grad()
def evaluate_loss(loader):
    model.eval()
    total_loss = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model(x)
        loss = criterion(out, y)
        total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
# =========================
# TRAINING LOOP WITH HISTORY
# =========================
best_val_loss = float('inf')
best_epoch = -1
early_stop_counter = 0

# Store metrics for plotting
train_losses = []
val_losses = []
val_accs = []

print("\n===== STARTING TRAINING =====\n")
for epoch in range(EPOCHS):
    start_time = time.time()

    train_loss = train_one_epoch()
    val_loss = evaluate_loss(val_loader)
    val_acc = evaluate_accuracy(val_loader)

    # Store history
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), SAVE_PATH)
        early_stop_counter = 0
        print(f"*** New best model saved (val_loss={val_loss:.4f}) ***")
    else:
        early_stop_counter += 1
        if early_stop_counter >= PATIENCE:
            print(f"Early stopping triggered after {epoch+1} epochs")
            break

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_acc:.4f} | "
          f"Time: {time.time()-start_time:.1f}s | "
          f"LR: {optimizer.param_groups[0]['lr']:.2e}")

print(f"\nTraining finished. Best val loss = {best_val_loss:.4f} at epoch {best_epoch+1}")


===== STARTING TRAINING =====



In [ ]:
# =========================
# PLOT LOSS & ACCURACY CURVES
# =========================
epochs_range = range(1, len(train_losses) + 1)

plt.figure(figsize=(12, 5))

# Loss curves
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_losses, 'b-', label='Training Loss')
plt.plot(epochs_range, val_losses, 'r-', label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)

# Accuracy curve
plt.subplot(1, 2, 2)
plt.plot(epochs_range, val_accs, 'g-', label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Validation Accuracy over Epochs')
plt.legend()
plt.grid(True)

plt.tight_layout()
loss_acc_plot_path = os.path.join(PLOTS_DIR, 'loss_accuracy_curves.png')
plt.savefig(loss_acc_plot_path, dpi=300)
print(f"✅ Loss & accuracy curves saved to {loss_acc_plot_path}")
plt.show()


In [ ]:

# =========================
# FINAL TEST EVALUATION & CONFUSION MATRIX
# =========================
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
test_acc = evaluate_accuracy(test_loader)
print(f"\n✅ Final test accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

# Get all predictions and true labels from test set
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(DEVICE)
        out = model(x)
        preds = torch.argmax(out, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y.numpy())

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)

# Plot confusion matrix as heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix on Test Set')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
cm_plot_path = os.path.join(PLOTS_DIR, 'confusion_matrix.png')
plt.savefig(cm_plot_path, dpi=300)
print(f"✅ Confusion matrix saved to {cm_plot_path}")
plt.show()

# Optional: classification report
print("\n📊 Classification Report:")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

In [ ]:
# =========================
# SAVE ADDITIONAL METRICS TO TEXT FILE (for report)
# =========================
report_path = os.path.join(PLOTS_DIR, 'training_summary.txt')
with open(report_path, 'w') as f:
    f.write("===== SKIN CONDITION CLASSIFICATION REPORT =====\n\n")
    f.write(f"Classes: {CLASS_NAMES}\n")
    f.write(f"Number of classes: {NUM_CLASSES}\n")
    f.write(f"Dataset split: Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}\n\n")
    f.write(f"Best validation loss: {best_val_loss:.4f} at epoch {best_epoch+1}\n")
    f.write(f"Final test accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)\n\n")
    f.write("Classification Report:\n")
    f.write(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))
print(f"✅ Training summary saved to {report_path}")


In [ ]:
# =========================
# INFERENCE FUNCTION (unchanged)
# =========================
def predict(image_path, model_state_path=SAVE_PATH):
    model.load_state_dict(torch.load(model_state_path, map_location=DEVICE))
    model.eval()
    img = Image.open(image_path).convert('RGB')
    img = val_test_transforms(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(img)
        probs = torch.softmax(logits, dim=1)
        pred_idx = torch.argmax(probs, dim=1).item()
    return CLASS_NAMES[pred_idx], probs.cpu().numpy()[0]